# Proyecto 2 - Modelado Predictivo de Mortalidad (Fases 0-5)

Este notebook implementa las primeras 5 fases del plan de prediccion y mantiene separado el Proyecto 1 (EDA/Clustering) del Proyecto 2 (Modelos).

Fases cubiertas aqui:
- Fase 0: Reutilizacion controlada
- Fase 1: Definir variable objetivo
- Fase 2: Antecedentes (plantilla academica)
- Fase 3: Preparacion de datos
- Fase 4: Train/Validation/Test split
- Fase 5: Seleccion de algoritmos


## Fase 0 - Reutilizacion controlada

Este notebook reutiliza logica de limpieza ya alineada con el Proyecto 1 (ejemplo: tratamiento de `Edadif == 999` y manejo de columnas entre anos).

Se mantiene separado de `main.ipynb` para:
- evitar notebooks demasiado largos,
- no afectar el flujo EDA/Clustering,
- facilitar revision independiente en GitHub y por el profesor.


In [40]:
from __future__ import annotations

from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
import pyreadstat

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

In [41]:
RANDOM_STATE = 42
YEARS = list(range(2013, 2023))
DATA_DIR = Path('data/defunciones')

# Opcional para prototipar rapido en equipos con menos RAM
USE_SAMPLE = False
SAMPLE_SIZE = 200_000

np.random.seed(RANDOM_STATE)

In [42]:
def normalize_text(value: str) -> str:
    if value is None:
        return ''
    value = unicodedata.normalize('NFKD', str(value))
    value = ''.join(ch for ch in value if not unicodedata.combining(ch))
    return value.lower().strip()


def resolve_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    norm_map = {normalize_text(col): col for col in df.columns}
    for candidate in candidates:
        key = normalize_text(candidate)
        if key in norm_map:
            return norm_map[key]
    return None


def normalize_cie10(code: object) -> str | None:
    if pd.isna(code):
        return None
    text = str(code).upper().strip()
    # Captura codigos tipo A15, I21, Y87.0
    match = re.search(r'[A-Z][0-9]{2}(?:\.[0-9])?', text)
    return match.group(0) if match else None


def _cie_num(cie10: str) -> int | None:
    try:
        return int(cie10[1:3])
    except Exception:
        return None


def map_causa_grupo(cie10: str | None) -> str | None:
    """
    Mapeo detallado inspirado en diccionario.md, con clases no traslapadas
    para clasificacion supervisada.
    """
    if cie10 is None:
        return None

    letter = cie10[0]
    num = _cie_num(cie10)

    # Infecciosas
    if letter in {'A', 'B'}:
        return 'Infecciosas'

    # Neoplasias / sangre e inmunidad
    if letter == 'C':
        return 'Neoplasias'
    if letter == 'D' and num is not None:
        if 0 <= num <= 48:
            return 'Neoplasias'
        if 50 <= num <= 89:
            return 'Sangre_inmunidad'

    # Endocrinas, mentales, nervioso/sentidos
    if letter == 'E':
        return 'Endocrinas_metabolicas'
    if letter == 'F':
        return 'Trastornos_mentales'
    if letter in {'G', 'H'}:
        return 'Nervioso_organos_sentidos'

    # Circulatorias (subgrupos del diccionario)
    if letter == 'I' and num is not None:
        if 10 <= num <= 13:
            return 'Hipertensiva'
        if 20 <= num <= 25:
            return 'Isquemica_corazon'
        if 60 <= num <= 69:
            return 'Cerebrovascular'
        return 'Otras_circulatorias'

    # Respiratorias (subgrupos del diccionario)
    if letter == 'J' and num is not None:
        if 10 <= num <= 18:
            return 'Neumonia_influenza'
        if 40 <= num <= 47:
            return 'EPOC'
        return 'Otras_respiratorias'

    # Digestivas (subgrupos del diccionario)
    if letter == 'K' and num is not None:
        if num == 70 or 73 <= num <= 74:
            return 'Cronica_higado'
        if 35 <= num <= 38:
            return 'Apendicitis'
        if (40 <= num <= 46) or num == 56:
            return 'Hernia_obstruccion_intestinal'
        return 'Otras_digestivas'

    # Genitourinarias (subgrupos del diccionario)
    if letter == 'N' and num is not None:
        if (0 <= num <= 7) or (17 <= num <= 19) or (25 <= num <= 27):
            return 'Nefritis_sindrome_nefrotico'
        return 'Otras_genitourinarias'

    # Causas externas (subgrupos del diccionario)
    if letter == 'V' and num is not None:
        if 2 <= num <= 89:
            return 'Accidentes_transito'
        return 'Accidentes_no_intencionales'

    if letter == 'W':
        return 'Accidentes_no_intencionales'

    if letter == 'X' and num is not None:
        if 60 <= num <= 84:
            return 'Suicidio'
        if 85 <= num <= 99:
            return 'Homicidio'
        return 'Accidentes_no_intencionales'

    if letter == 'Y' and num is not None:
        if 0 <= num <= 9:
            return 'Homicidio'
        if 40 <= num <= 59:
            return 'Efectos_adversos_medicamentos'
        if 85 <= num <= 86:
            return 'Accidentes_no_intencionales'
        return 'Otras_causas_externas'

    # Separaciones clave para no perder informacion
    if letter == 'R' and num is not None:
        if num == 98:
            return 'Muerte_sin_asistencia_R98'
        if num == 99:
            return 'Causa_mal_definida_R99'
        if num == 54:
            return 'Senilidad_R54'
        return 'Sintomas_signos_hallazgos'
    if letter == 'Q':
        return 'Congenitas'
    if letter == 'P':
        return 'Perinatales'
    if letter == 'O':
        return 'Embarazo_parto_puerperio'
    if letter == 'U':
        return 'Codigos_especiales'

    # Otros capitulos CIE-10
    if letter in {'L', 'M', 'Z'}:
        return 'Otros_capitulos'

    return 'Otros_capitulos'


def load_yearly_data(data_dir: Path, years: list[int]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for year in years:
        path = data_dir / f'{year}.sav'
        if not path.exists():
            print(f'[WARN] No existe: {path}')
            continue
        df_year, _ = pyreadstat.read_sav(str(path))
        df_year['__year_file__'] = year
        frames.append(df_year)
        print(f'[OK] {year}: {len(df_year):,} registros')

    if not frames:
        raise FileNotFoundError('No se pudieron cargar archivos .sav en data/defunciones')

    return pd.concat(frames, ignore_index=True)


In [43]:
df = load_yearly_data(DATA_DIR, YEARS)
print(f'\nShape consolidado: {df.shape}')

if USE_SAMPLE and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'Se usa muestra para prototipado: {df.shape}')

[OK] 2013: 76,639 registros
[OK] 2014: 77,807 registros
[OK] 2015: 80,876 registros
[OK] 2016: 82,565 registros
[OK] 2017: 81,726 registros
[OK] 2018: 83,071 registros
[OK] 2019: 85,600 registros
[OK] 2020: 96,001 registros
[OK] 2021: 118,465 registros
[OK] 2022: 95,386 registros

Shape consolidado: (878136, 30)


## Fase 1 - Variable objetivo

**Variable respuesta seleccionada:** `grupo_causa_cie10` (derivada de `Caudef` CIE-10).

- Tipo: **cualitativa nominal** (clasificacion multiclase).
- Enfoque: mapeo **detallado** inspirado en `diccionario.md`, no consolidado.
- Clases principales consideradas:
  - `Infecciosas`, `Neoplasias`, `Sangre_inmunidad`, `Endocrinas_metabolicas`, `Trastornos_mentales`, `Nervioso_organos_sentidos`
  - `Hipertensiva`, `Isquemica_corazon`, `Cerebrovascular`, `Otras_circulatorias`
  - `Neumonia_influenza`, `EPOC`, `Otras_respiratorias`
  - `Cronica_higado`, `Apendicitis`, `Hernia_obstruccion_intestinal`, `Otras_digestivas`
  - `Nefritis_sindrome_nefrotico`, `Otras_genitourinarias`
  - `Accidentes_transito`, `Accidentes_no_intencionales`, `Suicidio`, `Homicidio`, `Efectos_adversos_medicamentos`, `Otras_causas_externas`
  - `Muerte_sin_asistencia_R98`, `Causa_mal_definida_R99`, `Senilidad_R54`, `Sintomas_signos_hallazgos`
  - `Congenitas`, `Perinatales`, `Embarazo_parto_puerperio`, `Codigos_especiales`, `Otros_capitulos`

Fuente: `diccionario.md` (tabla CIE-10) y estructura de capitulos ICD-10 OMS.

Nota metodologica: las clases R pueden sugerir problemas de calidad de certificacion o diagnostico, pero no prueban por si solas negligencia medica.


In [44]:
caudef_col = resolve_column(df, ['Caudef'])
if caudef_col is None:
    raise KeyError('No se encontro la columna Caudef para construir la variable objetivo.')

df['cie10_norm'] = df[caudef_col].apply(normalize_cie10)
df['grupo_causa_cie10'] = df['cie10_norm'].apply(map_causa_grupo)

print('Distribucion inicial de la variable objetivo:')
display(df['grupo_causa_cie10'].value_counts(dropna=False).to_frame('conteo'))

df = df[df['grupo_causa_cie10'].notna()].copy()
print(f'\nRegistros tras remover objetivo nulo: {len(df):,}')

Distribucion inicial de la variable objetivo:


,conteo
grupo_causa_cie10,
Endocrinas_metabolicas,91789
Neoplasias,84843
Isquemica_corazon,68284
Neumonia_influenza,62336
Accidentes_no_intencionales,60300
Infecciosas,41420
Muerte_sin_asistencia_R98,34582
Homicidio,34397
Cronica_higado,34237



Registros tras remover objetivo nulo: 878,136


## Diagnostico de clases R y `Otros_capitulos`

Se analiza por separado `R98`, `R99` y `R54`, dejando `Sintomas_signos_hallazgos` para el resto de codigos R.


In [45]:
def resumen_clase(clase: str) -> None:
    mask = df['grupo_causa_cie10'].eq(clase)
    df_sub = df.loc[mask, ['cie10_norm', caudef_col, '__year_file__']].copy()

    print(f'=== {clase} ===')
    if df_sub.empty:
        print('Sin registros.')
        return

    total = len(df)
    n = len(df_sub)
    print(f'Registros: {n:,} ({n / total:.2%} del total)')

    df_sub['cie10_letra'] = df_sub['cie10_norm'].str[0]

    print()
    print('Top 12 codigos CIE-10:')
    display(df_sub['cie10_norm'].value_counts().head(12).to_frame('conteo'))

    print()
    print('Distribucion por letra CIE-10:')
    display(df_sub['cie10_letra'].value_counts().to_frame('conteo'))

clases_r = [
    'Muerte_sin_asistencia_R98',
    'Causa_mal_definida_R99',
    'Senilidad_R54',
    'Sintomas_signos_hallazgos',
    'Otros_capitulos',
]

for c in clases_r:
    resumen_clase(c)
    print()


=== Muerte_sin_asistencia_R98 ===
Registros: 34,582 (3.94% del total)

Top 12 codigos CIE-10:


,conteo
cie10_norm,
R98,34582



Distribucion por letra CIE-10:


,conteo
cie10_letra,
R,34582



=== Causa_mal_definida_R99 ===
Registros: 27,258 (3.10% del total)

Top 12 codigos CIE-10:


...

,conteo
cie10_letra,
R,17715



=== Otros_capitulos ===
Registros: 5,010 (0.57% del total)

Top 12 codigos CIE-10:


,conteo
cie10_norm,
M06,763
L98,618
M32,548
L89,354
M13,327
M19,323
M81,282
L08,243
M79,185



Distribucion por letra CIE-10:


,conteo
cie10_letra,
M,3352
L,1658


## Interpretacion de clases R (detalle)

Se separaron los codigos dominantes de R para no ocultar informacion:
- `R98`: muerte sin asistencia
- `R99`: causa mal definida/no especificada
- `R54`: senilidad
- `Sintomas_signos_hallazgos`: resto de codigos R

Lectura epidemiologica sugerida:
- Un volumen alto de estas clases puede reflejar limites de certificacion, registro o acceso diagnostico.
- No implica por si solo negligencia medica, pero si una señal fuerte de calidad del dato.


In [46]:
R_DESCRIPTIONS = {
    'R98': 'Muerte sin asistencia (unattended death)',
    'R99': 'Otras causas mal definidas o no especificadas de mortalidad',
    'R54': 'Senilidad',
    'R68': 'Otros sintomas y signos generales',
    'R50': 'Fiebre de origen desconocido',
    'R57': 'Choque (shock), no clasificado en otra parte',
    'R09': 'Otros sintomas del sistema circulatorio y respiratorio',
}

# Analiza solo el R residual (ya sin R98, R99, R54)
df_r_residual = df[df['grupo_causa_cie10'] == 'Sintomas_signos_hallazgos'].copy()

if df_r_residual.empty:
    print('No hay registros en Sintomas_signos_hallazgos residual.')
else:
    top_r = (
        df_r_residual['cie10_norm']
        .value_counts()
        .head(12)
        .rename_axis('codigo_r')
        .reset_index(name='conteo')
    )
    top_r['descripcion'] = top_r['codigo_r'].map(R_DESCRIPTIONS).fillna('Descripcion CIE-10 no cargada en este resumen')
    top_r['porcentaje_dentro_R_residual'] = (top_r['conteo'] / len(df_r_residual) * 100).round(2)

    print(f'Registros totales en R residual: {len(df_r_residual):,}')
    print('Top codigos R residuales y su interpretacion:')
    display(top_r[['codigo_r', 'conteo', 'porcentaje_dentro_R_residual', 'descripcion']])

# Resumen de los 4 grupos R
resumen_r = df['grupo_causa_cie10'].value_counts().reindex([
    'Muerte_sin_asistencia_R98',
    'Causa_mal_definida_R99',
    'Senilidad_R54',
    'Sintomas_signos_hallazgos',
]).fillna(0).astype(int).to_frame('conteo')

resumen_r['porcentaje_total'] = (resumen_r['conteo'] / len(df) * 100).round(2)
print()
print('Resumen de desagregacion de R:')
display(resumen_r)


Registros totales en R residual: 17,715
Top codigos R residuales y su interpretacion:


,codigo_r,conteo,porcentaje_dentro_R_residual,descripcion
0,R68,2688,15.17,Otros sintomas y signos generales
1,R50,2589,14.61,Fiebre de origen desconocido
2,R57,2393,13.51,"Choque (shock), no clasificado en otra parte"
3,R09,2358,13.31,Otros sintomas del sistema circulatorio y resp...
4,R95,1684,9.51,Descripcion CIE-10 no cargada en este resumen
5,R56,1666,9.40,Descripcion CIE-10 no cargada en este resumen
6,R10,1299,7.33,Descripcion CIE-10 no cargada en este resumen
7,R96,747,4.22,Descripcion CIE-10 no cargada en este resumen
8,R05,557,3.14,Descripcion CIE-10 no cargada en este resumen
9,R11,179,1.01,Descripcion CIE-10 no cargada en este resumen



Resumen de desagregacion de R:


,conteo,porcentaje_total
grupo_causa_cie10,,
Muerte_sin_asistencia_R98,34582,3.94
Causa_mal_definida_R99,27258,3.10
Senilidad_R54,23587,2.69
Sintomas_signos_hallazgos,17715,2.02


## Fase 3 - Preparacion de datos

En esta fase se construye una matriz de modelado sin fuga de informacion:
- no se usa `Caudef` como feature (origen directo de la etiqueta),
- se limpian sentinelas (`Edadif == 999`),
- se generan variables temporales simples para mejorar senal predictiva.


In [47]:
sexo_col = resolve_column(df, ['Sexo'])
edad_col = resolve_column(df, ['Edadif'])
depocu_col = resolve_column(df, ['Depocu', 'Depreg'])
mupocu_col = resolve_column(df, ['Mupocu', 'Mupreg'])
mes_col = resolve_column(df, ['Mesocu', 'Mesreg'])
dia_col = resolve_column(df, ['Diaocu'])
anio_col = resolve_column(df, ['Añoocu', 'Anoocu', 'Añoreg', 'Anoreg'])
ecivil_col = resolve_column(df, ['Ecidif'])
escolar_col = resolve_column(df, ['Escodif'])
ocup_col = resolve_column(df, ['Ciuodif'])

feature_map = {
    'sexo': sexo_col,
    'edad': edad_col,
    'departamento': depocu_col,
    'municipio': mupocu_col,
    'mes': mes_col,
    'dia': dia_col,
    'anio': anio_col,
    'estado_civil': ecivil_col,
    'escolaridad': escolar_col,
    'ocupacion': ocup_col,
}

missing_features = [k for k, v in feature_map.items() if v is None]
print('Columnas detectadas:')
display(pd.Series(feature_map, name='columna_en_dataset'))
if missing_features:
    print(f'[WARN] No detectadas: {missing_features}')

selected_pairs = [(k, v) for k, v in feature_map.items() if v is not None]
df_model = df[[v for _, v in selected_pairs]].copy()
df_model.columns = [k for k, _ in selected_pairs]

# Limpieza principal heredada de practicas del Proyecto 1
if 'edad' in df_model.columns:
    df_model['edad'] = pd.to_numeric(df_model['edad'], errors='coerce')
    df_model.loc[df_model['edad'] == 999, 'edad'] = np.nan

if 'dia' in df_model.columns:
    df_model['dia'] = pd.to_numeric(df_model['dia'], errors='coerce')

if 'mes' in df_model.columns:
    df_model['mes'] = pd.to_numeric(df_model['mes'], errors='coerce')

if 'anio' in df_model.columns:
    df_model['anio'] = pd.to_numeric(df_model['anio'], errors='coerce')

# Feature temporal simple
if {'dia', 'mes', 'anio'}.issubset(df_model.columns):
    fecha = pd.to_datetime(
        dict(year=df_model['anio'], month=df_model['mes'], day=df_model['dia']),
        errors='coerce',
    )
    df_model['es_fin_semana'] = fecha.dt.dayofweek.isin([5, 6]).astype('float')

y_raw = df['grupo_causa_cie10'].astype(str).copy()

MIN_CLASS_SHARE = 0.003  # 0.3%
class_share_raw = y_raw.value_counts(normalize=True)
rare_classes = class_share_raw[class_share_raw < MIN_CLASS_SHARE].index.tolist()

y = y_raw.where(~y_raw.isin(rare_classes), 'Otros_raros')

print(f'Shape de modelado: X={df_model.shape}, y={y.shape}')
print(f'Clases originales: {y_raw.nunique()} | Clases tras agrupacion de raras: {y.nunique()}')
print(f'Clases agrupadas en Otros_raros: {len(rare_classes)}')

Columnas detectadas:


sexo               Sexo
edad             Edadif
departamento     Depocu
municipio        Mupocu
mes              Mesocu
dia              Diaocu
anio             Añoocu
estado_civil     Ecidif
escolaridad     Escodif
ocupacion       Ciuodif
Name: columna_en_dataset, dtype: str

Shape de modelado: X=(878136, 11), y=(878136,)
Clases originales: 34 | Clases tras agrupacion de raras: 32
Clases agrupadas en Otros_raros: 3


In [48]:
print('Balance de clases global:')
display(y.value_counts(normalize=True).mul(100).round(2).to_frame('%'))

print('Nulos por variable (top 10):')
display(df_model.isna().mean().sort_values(ascending=False).head(10).to_frame('%_nulos'))

Balance de clases global:


,%
grupo_causa_cie10,
Endocrinas_metabolicas,10.45
Neoplasias,9.66
Isquemica_corazon,7.78
Neumonia_influenza,7.10
Accidentes_no_intencionales,6.87
Infecciosas,4.72
Muerte_sin_asistencia_R98,3.94
Homicidio,3.92
Cronica_higado,3.90


Nulos por variable (top 10):


,%_nulos
anio,0.175879
edad,0.005893
sexo,0.000000
departamento,0.000000
municipio,0.000000
mes,0.000000
dia,0.000000
estado_civil,0.000000
escolaridad,0.000000
ocupacion,0.000000


## Fase 4 - Train/Validation/Test split

Metodo seguido para construir entrenamiento y prueba:
1. Se realiza una primera particion de `df_model` y `y` con `train_test_split(..., test_size=0.30, stratify=y)`.
   - Resultado: 70% entrenamiento y 30% temporal.
2. El bloque temporal (30%) se divide de nuevo con `train_test_split(..., test_size=0.50, stratify=y_temp)`.
   - Resultado final: 15% validacion y 15% prueba.

Porcentajes esperados del total:
- Train: 70%
- Valid: 15%
- Test : 15%

Como la variable respuesta es categorica (`grupo_causa_cie10`), se evalua balance por clases entre particiones (no aplica analisis de atipicos de variable respuesta cuantitativa).


In [49]:
X_train, X_temp, y_train, y_temp = train_test_split(
    df_model,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print('Shapes:')
print(f'  Train: {X_train.shape} | {y_train.shape}')
print(f'  Valid: {X_valid.shape} | {y_valid.shape}')
print(f'  Test : {X_test.shape} | {y_test.shape}')

n_total = len(df_model)
pct_train = len(X_train) / n_total * 100
pct_valid = len(X_valid) / n_total * 100
pct_test = len(X_test) / n_total * 100

print('Porcentaje real por particion:')
print(f'  Train: {pct_train:.2f}%')
print(f'  Valid: {pct_valid:.2f}%')
print(f'  Test : {pct_test:.2f}%')

def class_share(series: pd.Series) -> pd.Series:
    return series.value_counts(normalize=True).mul(100).round(2)

balance = pd.concat(
    [
        class_share(y_train).rename('train_%'),
        class_share(y_valid).rename('valid_%'),
        class_share(y_test).rename('test_%'),
    ],
    axis=1,
).fillna(0.0)

max_gap_between_splits = (balance.max(axis=1) - balance.min(axis=1)).max()
global_share = y.value_counts(normalize=True).mul(100)
global_max = global_share.max()
global_min = global_share.min()
global_ratio = global_max / global_min

print('Diagnostico de balance:')
print(f'  Max diferencia de porcentaje por clase entre train/valid/test: {max_gap_between_splits:.2f} pp')
print(f'  Clase mas frecuente (global): {global_max:.2f}%')
print(f'  Clase menos frecuente (global): {global_min:.2f}%')
print(f'  Relacion max/min (global): {global_ratio:.1f}x')
print('  Conclusion: la distribucion entre particiones esta balanceada por estratificacion,')
print('  pero el problema multiclase global esta desbalanceado por clases raras.')

display(balance)

Shapes:
  Train: (614695, 11) | (614695,)
  Valid: (131720, 11) | (131720,)
  Test : (131721, 11) | (131721,)
Porcentaje real por particion:
  Train: 70.00%
  Valid: 15.00%
  Test : 15.00%
Diagnostico de balance:
  Max diferencia de porcentaje por clase entre train/valid/test: 0.00 pp
  Clase mas frecuente (global): 10.45%
  Clase menos frecuente (global): 0.14%
  Relacion max/min (global): 75.1x
  Conclusion: la distribucion entre particiones esta balanceada por estratificacion,
  pero el problema multiclase global esta desbalanceado por clases raras.


,train_%,valid_%,test_%
grupo_causa_cie10,,,
Endocrinas_metabolicas,10.45,10.45,10.45
Neoplasias,9.66,9.66,9.66
Isquemica_corazon,7.78,7.78,7.78
Neumonia_influenza,7.10,7.10,7.10
Accidentes_no_intencionales,6.87,6.87,6.87
Infecciosas,4.72,4.72,4.72
Muerte_sin_asistencia_R98,3.94,3.94,3.94
Homicidio,3.92,3.92,3.92
Cronica_higado,3.90,3.90,3.90


## Fase 6-7 - Modelos supervisados y seleccion del mejor

Se probaran tres familias de algoritmos solicitadas:
- Decision Tree
- Random Forest
- XGBoost

Preprocesamiento aplicado para entrenar correctamente:
1. Variables numericas: imputacion por mediana.
2. Variables categoricas: imputacion por moda + One-Hot Encoding (`handle_unknown='ignore'`).
3. Clases extremadamente raras de la variable respuesta se agrupan en `Otros_raros` para estabilizar entrenamiento y mejorar capacidad predictiva.
4. Todo se encapsula en `Pipeline` + `ColumnTransformer` para evitar fuga de informacion.

Se entrenan al menos 3 variantes por algoritmo (9 modelos en total), cambiando hiperparametros para buscar mejor generalizacion sin subajuste/sobreajuste.

Criterio de seleccion:
- metrica principal: `F1 macro` en validacion (adecuada por desbalance multiclase),
- apoyo: `F1 weighted`, `accuracy` y brecha `train-valid` para detectar sobreajuste.


In [50]:
import json
import time
from datetime import datetime

import joblib
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError(
        'No se encontro xgboost. Instala con: pip install xgboost'
    ) from exc

In [51]:
num_features = X_train.select_dtypes(include=[np.number, 'bool']).columns.tolist()
cat_features = [c for c in X_train.columns if c not in num_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_features),
        (
            'cat',
            Pipeline(
                steps=[
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore')),
                ]
            ),
            cat_features,
        ),
    ],
    remainder='drop',
)

NUM_MODELING_THREADS = 4
SELECTION_SAMPLE_SIZE = 300_000

if len(X_train) > SELECTION_SAMPLE_SIZE:
    X_train_eval, _, y_train_eval, _ = train_test_split(
        X_train,
        y_train,
        train_size=SELECTION_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )
else:
    X_train_eval, y_train_eval = X_train, y_train

print('Configuracion de presupuesto de entrenamiento:')
print(f'  Muestras para seleccionar 9 modelos: {len(X_train_eval):,}')
print(f'  Hilos por modelo: {NUM_MODELING_THREADS}')

label_encoder = LabelEncoder()
label_encoder.fit(y_train.astype(str))
y_train_eval_enc = label_encoder.transform(y_train_eval.astype(str))

sample_weight_eval = compute_sample_weight(class_weight='balanced', y=y_train_eval)
sample_weight_eval_enc = compute_sample_weight(class_weight='balanced', y=y_train_eval_enc)

num_classes = y_train.nunique()

candidate_models = [
    {
        'modelo_id': 'dt_1',
        'algoritmo': 'DecisionTree',
        'estimator': DecisionTreeClassifier(
            criterion='gini',
            max_depth=10,
            min_samples_leaf=15,
            random_state=RANDOM_STATE,
        ),
    },
    {
        'modelo_id': 'dt_2',
        'algoritmo': 'DecisionTree',
        'estimator': DecisionTreeClassifier(
            criterion='entropy',
            max_depth=16,
            min_samples_leaf=8,
            random_state=RANDOM_STATE,
        ),
    },
    {
        'modelo_id': 'dt_3',
        'algoritmo': 'DecisionTree',
        'estimator': DecisionTreeClassifier(
            criterion='gini',
            max_depth=8,
            min_samples_leaf=25,
            class_weight='balanced',
            random_state=RANDOM_STATE,
        ),
    },
    {
        'modelo_id': 'rf_1',
        'algoritmo': 'RandomForest',
        'estimator': RandomForestClassifier(
            n_estimators=180,
            max_depth=18,
            min_samples_leaf=3,
            class_weight='balanced_subsample',
            n_jobs=NUM_MODELING_THREADS,
            random_state=RANDOM_STATE,
        ),
    },
    {
        'modelo_id': 'rf_2',
        'algoritmo': 'RandomForest',
        'estimator': RandomForestClassifier(
            n_estimators=260,
            max_depth=24,
            min_samples_leaf=2,
            class_weight='balanced_subsample',
            n_jobs=NUM_MODELING_THREADS,
            random_state=RANDOM_STATE,
        ),
    },
    {
        'modelo_id': 'rf_3',
        'algoritmo': 'RandomForest',
        'estimator': RandomForestClassifier(
            n_estimators=320,
            max_depth=None,
            min_samples_leaf=2,
            class_weight='balanced',
            n_jobs=NUM_MODELING_THREADS,
            random_state=RANDOM_STATE,
        ),
    },
    {
        'modelo_id': 'xgb_1',
        'algoritmo': 'XGBoost',
        'estimator': XGBClassifier(
            objective='multi:softprob',
            num_class=num_classes,
            n_estimators=220,
            learning_rate=0.08,
            max_depth=7,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='mlogloss',
            tree_method='hist',
            n_jobs=NUM_MODELING_THREADS,
            random_state=RANDOM_STATE,
        ),
    },
    {
        'modelo_id': 'xgb_2',
        'algoritmo': 'XGBoost',
        'estimator': XGBClassifier(
            objective='multi:softprob',
            num_class=num_classes,
            n_estimators=300,
            learning_rate=0.06,
            max_depth=9,
            subsample=0.85,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            eval_metric='mlogloss',
            tree_method='hist',
            n_jobs=NUM_MODELING_THREADS,
            random_state=RANDOM_STATE,
        ),
    },
    {
        'modelo_id': 'xgb_3',
        'algoritmo': 'XGBoost',
        'estimator': XGBClassifier(
            objective='multi:softprob',
            num_class=num_classes,
            n_estimators=380,
            learning_rate=0.05,
            max_depth=9,
            subsample=0.8,
            colsample_bytree=0.75,
            min_child_weight=3,
            reg_lambda=3.0,
            eval_metric='mlogloss',
            tree_method='hist',
            n_jobs=NUM_MODELING_THREADS,
            random_state=RANDOM_STATE,
        ),
    },
]

results = []
fitted_models = {}

for i, cfg in enumerate(candidate_models, start=1):
    print(f"[{i}/{len(candidate_models)}] Entrenando {cfg['modelo_id']} ({cfg['algoritmo']})...")
    is_xgb = cfg['algoritmo'] == 'XGBoost'
    pipe = Pipeline(
        steps=[
            ('preprocess', preprocessor),
            ('model', cfg['estimator']),
        ]
    )

    y_fit = y_train_eval_enc if is_xgb else y_train_eval
    fit_params = {}
    if cfg['algoritmo'] == 'RandomForest':
        fit_params['model__sample_weight'] = sample_weight_eval
    elif cfg['algoritmo'] == 'XGBoost':
        fit_params['model__sample_weight'] = sample_weight_eval_enc

    t0 = time.time()
    pipe.fit(X_train_eval, y_fit, **fit_params)
    fit_seconds = time.time() - t0

    pred_train_raw = pipe.predict(X_train_eval)
    pred_valid_raw = pipe.predict(X_valid)
    if is_xgb:
        pred_train = label_encoder.inverse_transform(pred_train_raw.astype(int))
        pred_valid = label_encoder.inverse_transform(pred_valid_raw.astype(int))
    else:
        pred_train = pred_train_raw
        pred_valid = pred_valid_raw

    f1_macro_train = f1_score(y_train_eval, pred_train, average='macro')
    f1_macro_valid = f1_score(y_valid, pred_valid, average='macro')
    f1_weighted_valid = f1_score(y_valid, pred_valid, average='weighted')
    acc_train = accuracy_score(y_train_eval, pred_train)
    acc_valid = accuracy_score(y_valid, pred_valid)

    gap = f1_macro_train - f1_macro_valid
    if gap > 0.08:
        ajuste = 'posible_sobreajuste'
    elif f1_macro_train < 0.45 and f1_macro_valid < 0.45:
        ajuste = 'posible_subajuste'
    else:
        ajuste = 'ajuste_aceptable'

    results.append(
        {
            'modelo_id': cfg['modelo_id'],
            'algoritmo': cfg['algoritmo'],
            'f1_macro_train': round(f1_macro_train, 4),
            'f1_macro_valid': round(f1_macro_valid, 4),
            'f1_weighted_valid': round(f1_weighted_valid, 4),
            'acc_train': round(acc_train, 4),
            'acc_valid': round(acc_valid, 4),
            'gap_train_valid': round(gap, 4),
            'ajuste': ajuste,
            'fit_seconds': round(fit_seconds, 1),
            'params': cfg['estimator'].get_params(),
        }
    )
    fitted_models[cfg['modelo_id']] = pipe

results_df = pd.DataFrame(results).sort_values(
    by=['f1_macro_valid', 'f1_weighted_valid', 'acc_valid'],
    ascending=False,
).reset_index(drop=True)

print('Ranking de modelos (ordenado por F1 macro de validacion):')
display(
    results_df[
        [
            'modelo_id',
            'algoritmo',
            'f1_macro_train',
            'f1_macro_valid',
            'f1_weighted_valid',
            'acc_valid',
            'gap_train_valid',
            'ajuste',
            'fit_seconds',
        ]
    ]
)

print('Resumen por algoritmo (mejor F1 macro en validacion):')
display(results_df.groupby('algoritmo', as_index=False)['f1_macro_valid'].max())

Configuracion de presupuesto de entrenamiento:
  Muestras para seleccionar 9 modelos: 300,000
  Hilos por modelo: 4
[1/9] Entrenando dt_1 (DecisionTree)...
[2/9] Entrenando dt_2 (DecisionTree)...
[3/9] Entrenando dt_3 (DecisionTree)...
[4/9] Entrenando rf_1 (RandomForest)...
[5/9] Entrenando rf_2 (RandomForest)...
[6/9] Entrenando rf_3 (RandomForest)...
[7/9] Entrenando xgb_1 (XGBoost)...
[8/9] Entrenando xgb_2 (XGBoost)...
[9/9] Entrenando xgb_3 (XGBoost)...
Ranking de modelos (ordenado por F1 macro de validacion):


,modelo_id,algoritmo,f1_macro_train,f1_macro_valid,f1_weighted_valid,acc_valid,gap_train_valid,ajuste,fit_seconds
0,xgb_3,XGBoost,0.2615,0.1729,0.2003,0.2145,0.0886,posible_sobreajuste,180.3
1,xgb_2,XGBoost,0.2867,0.1723,0.2013,0.2152,0.1144,posible_sobreajuste,164.1
2,xgb_1,XGBoost,0.2447,0.1694,0.1966,0.2116,0.0754,posible_subajuste,101.8
3,dt_2,DecisionTree,0.2200,0.1393,0.1963,0.2248,0.0808,posible_sobreajuste,11.5
4,dt_1,DecisionTree,0.1229,0.1172,0.1798,0.2287,0.0057,posible_subajuste,5.1
5,rf_3,RandomForest,0.4420,0.1058,0.0960,0.1129,0.3362,posible_sobreajuste,370.1
6,dt_3,DecisionTree,0.0848,0.0810,0.0850,0.1281,0.0038,posible_subajuste,2.7
7,rf_2,RandomForest,0.0891,0.0348,0.0211,0.0271,0.0543,posible_subajuste,87.8
8,rf_1,RandomForest,0.0330,0.0183,0.0067,0.0112,0.0147,posible_subajuste,29.9


Resumen por algoritmo (mejor F1 macro en validacion):


,algoritmo,f1_macro_valid
0,DecisionTree,0.1393
1,RandomForest,0.1058
2,XGBoost,0.1729


In [53]:
if results_df.empty:
    raise RuntimeError('No se pudo entrenar ningun modelo; revisa errores previos de entrenamiento.')

best_row = results_df.iloc[0]
best_model_id = best_row['modelo_id']
best_algoritmo = best_row['algoritmo']

print(f"Mejor modelo en validacion: {best_model_id} ({best_algoritmo})")
print(f"F1 macro validacion: {best_row['f1_macro_valid']:.4f}")
print(f"Brecha train-valid: {best_row['gap_train_valid']:.4f}")

X_train_full = pd.concat([X_train, X_valid], axis=0)
y_train_full = pd.concat([y_train, y_valid], axis=0)

if len(X_train_full) > 550_000:
    X_train_final, _, y_train_final, _ = train_test_split(
        X_train_full,
        y_train_full,
        train_size=550_000,
        random_state=RANDOM_STATE,
        stratify=y_train_full,
    )
    print('Entrenamiento final acelerado para mantener tiempo objetivo (~30 min):')
    print(f'  Muestras usadas en modelo final: {len(X_train_final):,}')
else:
    X_train_final, y_train_final = X_train_full, y_train_full

best_estimator = clone(fitted_models[best_model_id].named_steps['model'])
sample_weight_final = compute_sample_weight(class_weight='balanced', y=y_train_final)
final_model = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', best_estimator),
    ]
)

is_best_xgb = best_algoritmo == 'XGBoost'
y_train_final_fit = label_encoder.transform(y_train_final.astype(str)) if is_best_xgb else y_train_final
sample_weight_final_enc = compute_sample_weight(class_weight='balanced', y=y_train_final_fit) if is_best_xgb else None

final_fit_params = {}
if best_algoritmo == 'RandomForest':
    final_fit_params['model__sample_weight'] = sample_weight_final
elif best_algoritmo == 'XGBoost':
    final_fit_params['model__sample_weight'] = sample_weight_final_enc

final_model.fit(X_train_final, y_train_final_fit, **final_fit_params)
pred_test_raw = final_model.predict(X_test)
pred_test = label_encoder.inverse_transform(pred_test_raw.astype(int)) if is_best_xgb else pred_test_raw

f1_macro_test = f1_score(y_test, pred_test, average='macro')
f1_weighted_test = f1_score(y_test, pred_test, average='weighted')
acc_test = accuracy_score(y_test, pred_test)

print('Metricas finales en TEST:')
print(f'  F1 macro   : {f1_macro_test:.4f}')
print(f'  F1 weighted: {f1_weighted_test:.4f}')
print(f'  Accuracy   : {acc_test:.4f}')

ARTIFACTS_DIR = Path('artifacts/modelos')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

model_path = ARTIFACTS_DIR / 'best_model.joblib'
meta_path = ARTIFACTS_DIR / 'best_model_metadata.json'
ranking_path = ARTIFACTS_DIR / 'model_ranking.csv'

joblib.dump(final_model, model_path)
results_df.to_csv(ranking_path, index=False)

metadata = {
    'saved_at': datetime.utcnow().isoformat() + 'Z',
    'best_model_id': best_model_id,
    'algorithm': best_algoritmo,
    'selection_metric': 'f1_macro_valid',
    'selection_train_size': int(len(X_train_eval)),
    'final_train_size': int(len(X_train_final)),
    'validation_metrics': {
        'f1_macro_valid': float(best_row['f1_macro_valid']),
        'f1_weighted_valid': float(best_row['f1_weighted_valid']),
        'acc_valid': float(best_row['acc_valid']),
        'gap_train_valid': float(best_row['gap_train_valid']),
    },
    'test_metrics': {
        'f1_macro_test': float(f1_macro_test),
        'f1_weighted_test': float(f1_weighted_test),
        'acc_test': float(acc_test),
    },
    'top_candidates': results_df.head(9).to_dict(orient='records'),
    'model_file': str(model_path),
    'ranking_file': str(ranking_path),
}

with meta_path.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f'Modelo persistido en: {model_path}')
print(f'Metadata persistida en: {meta_path}')
print(f'Ranking persistido en: {ranking_path}')

loaded_model = joblib.load(model_path)
sample_pred_raw = loaded_model.predict(X_test.head(5))
sample_pred = label_encoder.inverse_transform(sample_pred_raw.astype(int)) if is_best_xgb else sample_pred_raw
print('Predicciones de verificacion (modelo recargado, primeras 5 filas de test):')
print(sample_pred)

Mejor modelo en validacion: xgb_3 (XGBoost)
F1 macro validacion: 0.1729
Brecha train-valid: 0.0886
Entrenamiento final acelerado para mantener tiempo objetivo (~30 min):
  Muestras usadas en modelo final: 550,000
Metricas finales en TEST:
  F1 macro   : 0.1739
  F1 weighted: 0.2017
  Accuracy   : 0.2161


C:\Users\dijol\AppData\Local\Temp\ipykernel_19672\3632647179.py:71: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'saved_at': datetime.utcnow().isoformat() + 'Z',


Modelo persistido en: artifacts\modelos\best_model.joblib
Metadata persistida en: artifacts\modelos\best_model_metadata.json
Ranking persistido en: artifacts\modelos\model_ranking.csv
Predicciones de verificacion (modelo recargado, primeras 5 filas de test):
['Causa_mal_definida_R99' 'Perinatales' 'Otras_genitourinarias'
 'Muerte_sin_asistencia_R98' 'Embarazo_parto_puerperio']
